# Week 01 · Day 02 — Embeddings & Semantic Search

**Topics:** Embedding models · Cosine similarity · In-memory semantic search · Semantic search pipeline


In [ ]:
%pip install -q openai numpy

In [ ]:
import os
import numpy as np
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 1 · Getting Embeddings

An embedding is a list of floats (a vector) that captures the semantic meaning of text. Similar texts have vectors that point in similar directions.

In [ ]:
def embed(texts: list[str], model: str = "text-embedding-3-small") -> np.ndarray:
    response = client.embeddings.create(input=texts, model=model)
    return np.array([item.embedding for item in response.data])

sample_texts = [
    "The cat sat on the mat.",
    "A feline rested on the rug.",   # semantically similar to above
    "Machine learning transforms data into predictions.",
]

vectors = embed(sample_texts)
print(f"Embedding shape: {vectors.shape}")
print(f"First 5 dimensions of text 1: {vectors[0][:5].tolist()}")

## 2 · Cosine Similarity from Scratch

Cosine similarity measures the angle between two vectors, not their magnitude. Range: −1 (opposite) to 1 (identical). Typically 0.7+ for semantically similar texts.

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Compare all pairs
labels = ["cat mat", "feline rug", "ML data"]
for i in range(len(vectors)):
    for j in range(i + 1, len(vectors)):
        sim = cosine_similarity(vectors[i], vectors[j])
        print(f"{labels[i]:12s} ↔ {labels[j]:12s}: {sim:.4f}")

In [ ]:
# Visualizing similarity with a heatmap (matplotlib optional)
try:
    import matplotlib.pyplot as plt

    n = len(vectors)
    sim_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            sim_matrix[i, j] = cosine_similarity(vectors[i], vectors[j])

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(sim_matrix, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=30)
    ax.set_yticklabels(labels)
    plt.colorbar(im)
    plt.title("Cosine Similarity Matrix")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — skipping heatmap")

## 3 · In-Memory Semantic Search Engine

Build a search engine without a vector database: embed all documents once, then retrieve by comparing query embedding to all document embeddings.

In [ ]:
# Knowledge base — in a real system, embed once and cache
documents = [
    "Python is a high-level programming language known for its simplicity.",
    "FastAPI is a modern web framework for building APIs with Python.",
    "PostgreSQL is an open-source relational database system.",
    "Docker containers package applications with their dependencies.",
    "Kubernetes orchestrates containerized applications at scale.",
    "Redis is an in-memory data structure store used as a cache.",
    "NumPy provides efficient numerical computing in Python.",
    "Pandas is a data analysis and manipulation library.",
]

doc_embeddings = embed(documents)
print(f"Indexed {len(documents)} documents ({doc_embeddings.shape[1]}-dimensional)")

In [ ]:
def semantic_search(query: str, top_k: int = 3) -> list[tuple[float, str]]:
    query_vec = embed([query])[0]
    scores = [
        cosine_similarity(query_vec, doc_vec)
        for doc_vec in doc_embeddings
    ]
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    return [(score, documents[i]) for i, score in ranked[:top_k]]

queries = [
    "What's good for caching?",
    "running apps in containers",
    "Python data science tools",
]

for query in queries:
    print(f"Query: {query}")
    for score, doc in semantic_search(query):
        print(f"  {score:.4f} — {doc[:70]}")
    print()

## 4 · Embedding Model Comparison

| Model | Dimensions | Cost per 1M tokens | Notes |
|-------|-----------|-------------------|-------|
| `text-embedding-3-small` | 1536 | $0.020 | Best price/quality ratio |
| `text-embedding-3-large` | 3072 | $0.130 | Use for highest accuracy |
| `text-embedding-ada-002` | 1536 | $0.100 | Legacy — use 3-small instead |

In [ ]:
# text-embedding-3-small supports dimension reduction via 'dimensions' param
# Reducing dimensions lowers storage cost with minor quality tradeoff
r_full = client.embeddings.create(input=["test"], model="text-embedding-3-small")
r_small = client.embeddings.create(input=["test"], model="text-embedding-3-small", dimensions=256)

print(f"Full dimensions:    {len(r_full.data[0].embedding)}")
print(f"Reduced dimensions: {len(r_small.data[0].embedding)}")

## 5 · Batch Embedding (Efficient)

Always batch your embedding requests — one API call for N texts is far cheaper and faster than N API calls.

In [ ]:
import time

texts = [f"Document number {i}: some content about topic {i % 5}" for i in range(20)]

# Efficient: single batch call
t0 = time.perf_counter()
batch_result = client.embeddings.create(input=texts, model="text-embedding-3-small")
batch_time = time.perf_counter() - t0

print(f"Batch call: {len(batch_result.data)} embeddings in {batch_time:.2f}s")
print(f"Tokens used: {batch_result.usage.total_tokens}")

# For very large corpora, chunk into batches of ~2048 texts
BATCH_SIZE = 2048
def embed_large(texts: list[str]) -> np.ndarray:
    all_embeddings = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i : i + BATCH_SIZE]
        result = client.embeddings.create(input=batch, model="text-embedding-3-small")
        all_embeddings.extend([item.embedding for item in result.data])
    return np.array(all_embeddings)

print("embed_large() defined — handles corpora of any size")

## Summary

- Embeddings map text to a high-dimensional vector. Similar texts → similar vectors.
- Cosine similarity: `dot(a, b) / (|a| * |b|)`. Range [−1, 1].
- In-memory search: embed all docs once, compare query against them at runtime.
- Always batch embedding calls — one call for N texts, not N calls.
- `text-embedding-3-small` is the default choice for most use cases.

**Next:** [RAG Basics](week01-day03-rag-basics.ipynb)